# GDBFNet — resultados unificados (PC-GITA y NeuroVoz)

Notebook **único** que calcula, para los dos corpus y con un solo método coherente,
todos los resultados que aparecen en la memoria. Elimina las inconsistencias que
provenían de números escritos a mano en los notebooks de figuras.

**Todo es idéntico para ambos corpus:**
- Arquitectura: `DualBranchFusionNet` (GDBFNet, con compuerta de fusión $\alpha$).
- Entrenamiento: `train_fold_regularized` (codificadores congelados).
- Métricas: las cuatro (AUC, accuracy, sensibilidad, especificidad) en el **mismo**
  umbral de Youden, a nivel de sujeto.
- Estimación robusta: validación cruzada repetida + IC 95 % por *bootstrap*.
- Figuras (ROC y matriz de confusión) con **fondo blanco** académico.

Lo único que cambia entre corpus es la carga de datos. Al final se imprime un resumen
conjunto listo para transcribir a las tablas de la memoria.

In [25]:
# =============================== CONFIGURACIÓN ===============================
import os, sys, warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = "/Users/napster/Documents/upm"
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
IMAGES_DIR = os.path.join(PROJECT_ROOT, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

import numpy as np, torch
import matplotlib as mpl

mpl.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "text.color": "black",
        "axes.labelcolor": "black",
        "axes.edgecolor": "black",
        "xtick.color": "black",
        "ytick.color": "black",
        "font.family": "DejaVu Sans",
    }
)

CORPORA = ["pcgita", "neurovoz"]
TARGET_SR = 16_000
RANDOM_STATE = 42
N_FOLDS = 5
N_REPEATS = 10
N_TRIALS = 60
QUICK = False
if QUICK:
    N_TRIALS, N_REPEATS = 12, 3
EPOCHS = 40 if QUICK else 120
PATIENCE = 10 if QUICK else 20

# NeuroVoz
NV_ROOT = "data/neurovoz"
NV_AUDIO = f"{NV_ROOT}/audios"
NV_HC = f"{NV_ROOT}/metadata/metadata_hc.csv"
NV_PD = f"{NV_ROOT}/metadata/metadata_pd.csv"
NV_TASK = "PATAKA"

DEVICE = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
CACHE = os.path.join(PROJECT_ROOT, "nested_cache")
os.makedirs(CACHE, exist_ok=True)
FIGNAME = {
    "pcgita": ("res_roc.png", "res_confusion.png"),
    "neurovoz": ("res_roc_nv.png", "res_confusion_nv.png"),
}
print("device:", DEVICE, "| corpus:", CORPORA)

device: mps | corpus: ['pcgita', 'neurovoz']


## 1 — Bloques compartidos (arquitectura, entrenamiento, métricas)

In [26]:
# ================= MODELO / ENTRENAMIENTO / MÉTRICAS (compartido) =================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    roc_curve,
    confusion_matrix,
)
from src.models import DualBranchFusionNet  # GDBFNet con compuerta


class DualBranchDataset(Dataset):
    def __init__(self, t, s, l, idx):
        self.t = torch.tensor(t[idx], dtype=torch.float32)
        self.s = torch.tensor(s[idx], dtype=torch.float32)
        self.y = torch.tensor(l[idx], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.t[i], self.s[i], self.y[i]


def train_fold_regularized(
    train_ds,
    val_ds,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    label_smoothing=0.0,
    dropout=0.3,
    dim_proj=128,
    dim_hidden=64,
    epochs=120,
    patience=20,
    batch_size=8,
    scheduler_factor=0.5,
    scheduler_patience=7,
):
    tr = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    va = DataLoader(val_ds, batch_size=len(val_ds))
    model = DualBranchFusionNet(
        dim_proj=dim_proj, dim_hidden=dim_hidden, dropout=dropout
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=scheduler_factor, patience=scheduler_patience
    )
    best, bad, state = float("inf"), 0, None
    for _ in range(epochs):
        model.train()
        for z_t, z_s, y in tr:
            z_t, z_s, y = z_t.to(device), z_s.to(device), y.to(device)
            if label_smoothing > 0:
                y = y * (1 - label_smoothing) + label_smoothing / 2
            logits, _ = model(z_t, z_s)
            loss = crit(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            z_t, z_s, y = next(iter(va))
            z_t, z_s, y = z_t.to(device), z_s.to(device), y.to(device)
            logits, _ = model(z_t, z_s)
            vloss = crit(logits, y).item()
        sched.step(vloss)
        if vloss < best:
            best, bad, state = (
                vloss,
                0,
                {k: v.clone() for k, v in model.state_dict().items()},
            )
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        z_t, z_s, y = next(iter(va))
        z_t, z_s, y = z_t.to(device), z_s.to(device), y.to(device)
        logits, alphas = model(z_t, z_s)
        return (
            y.cpu().numpy(),
            torch.sigmoid(logits).cpu().numpy(),
            alphas.cpu().numpy(),
        )


def fit_model(
    train_ds,
    val_ds,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    label_smoothing=0.0,
    dropout=0.3,
    dim_proj=128,
    dim_hidden=64,
    epochs=120,
    patience=20,
    batch_size=8,
    scheduler_factor=0.5,
    scheduler_patience=7,
):
    """Entrena y devuelve el modelo (para serializar el modelo final)."""
    tr = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    va = DataLoader(val_ds, batch_size=len(val_ds))
    model = DualBranchFusionNet(
        dim_proj=dim_proj, dim_hidden=dim_hidden, dropout=dropout
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=scheduler_factor, patience=scheduler_patience
    )
    best, bad, state = float("inf"), 0, None
    for _ in range(epochs):
        model.train()
        for z_t, z_s, y in tr:
            z_t, z_s, y = z_t.to(device), z_s.to(device), y.to(device)
            if label_smoothing > 0:
                y = y * (1 - label_smoothing) + label_smoothing / 2
            logits, _ = model(z_t, z_s)
            loss = crit(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            z_t, z_s, y = next(iter(va))
            z_t, z_s, y = z_t.to(device), z_s.to(device), y.to(device)
            vloss = crit(model(z_t, z_s)[0], y).item()
        sched.step(vloss)
        if vloss < best:
            best, bad, state = (
                vloss,
                0,
                {k: v.clone() for k, v in model.state_dict().items()},
            )
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(state)
    model.eval()
    return model


def manifold_mixup_intraclass(t, s, l, alpha, rng):
    at, as_, al = [], [], []
    for cls in [0, 1]:
        ci = np.where(l == cls)[0]
        if len(ci) < 2:
            continue
        for i in ci:
            j = rng.choice(ci[ci != i])
            lam = rng.beta(alpha, alpha)
            at.append(lam * t[i] + (1 - lam) * t[j])
            as_.append(lam * s[i] + (1 - lam) * s[j])
            al.append(cls)
    return np.array(at), np.array(as_), np.array(al)


def youden_metrics(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    th = thr[np.argmax(tpr - fpr)]
    yp = (y_prob >= th).astype(int)
    return dict(
        auc=roc_auc_score(y_true, y_prob),
        acc=accuracy_score(y_true, yp),
        sens=recall_score(y_true, yp, pos_label=1, zero_division=0),
        spec=recall_score(y_true, yp, pos_label=0, zero_division=0),
        thr=float(th),
    )


def make_train_ds(T, S, L, tr, use_icmm, ma, seed):
    t, s, l = T[tr], S[tr], L[tr]
    if use_icmm:
        rng = np.random.default_rng(seed)
        at, as_, al = manifold_mixup_intraclass(t, s, l, ma, rng)
        t = np.concatenate([t, at])
        s = np.concatenate([s, as_])
        l = np.concatenate([l, al])
    return DualBranchDataset(t, s, l, np.arange(len(l)))


def bootstrap_ci(v, n_boot=1000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    v = np.asarray(v)
    b = [rng.choice(v, len(v), replace=True).mean() for _ in range(n_boot)]
    return (
        v.mean(),
        np.percentile(b, (100 - ci) / 2),
        np.percentile(b, 100 - (100 - ci) / 2),
    )

## 2 — Carga de datos por corpus (con cache)

In [27]:
# =========================== EMBEDDINGS POR CORPUS ===========================
from sklearn.preprocessing import StandardScaler


def prepare_pcgita():
    cf = os.path.join(CACHE, "pcgita_all.npz")
    if os.path.exists(cf):
        d = np.load(cf)
        return d["emb_t"], d["emb_s"], d["labels"].astype(np.float32)
    from src.preprocessing import load_waveforms, preprocess_waveform, load_metadata
    from src.embeddings import extract_multilayer_embeddings
    from src.spectral import extract_spectral_features

    df = load_metadata(base_path="data")
    wr, df = load_waveforms(df, target_sr=TARGET_SR)
    wp = [preprocess_waveform(w) for w in wr]
    emb = extract_multilayer_embeddings(wp, "facebook/wav2vec2-base", [0], DEVICE)[0]
    spec = extract_spectral_features(
        wp, device=DEVICE, sample_rate=TARGET_SR, n_fft=1024, hop_length=512, n_mels=128
    )
    T = StandardScaler().fit_transform(emb)
    S = StandardScaler().fit_transform(spec)
    L = df["label"].to_numpy()
    np.savez(cf, emb_t=T, emb_s=S, labels=L)
    return T, S, L.astype(np.float32)


def prepare_neurovoz():
    cf = os.path.join(CACHE, "neurovoz_all.npz")
    if os.path.exists(cf):
        d = np.load(cf)
        return d["emb_t"], d["emb_s"], d["labels"].astype(np.float32)
    import pandas as pd, soundfile as sf
    from math import gcd
    from scipy.signal import resample_poly
    from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
    from src.spectral import extract_spectral_features

    hc = pd.read_csv(NV_HC)
    pdd = pd.read_csv(NV_PD)
    df = pd.concat([hc, pdd], ignore_index=True)
    df["filename"] = df["Audio"].str.split("/").str[-1]
    df["task"] = df["filename"].str.extract(r"^[A-Z]+_(.+)_\d+\.wav")
    df = df[df["task"] == NV_TASK].copy()
    df["label"] = (df["Group"] == "PD").astype(int)
    df["path"] = df["filename"].apply(lambda f: os.path.join(NV_AUDIO, f))

    def load(p):
        w, sr = sf.read(p, dtype="float32", always_2d=False)
        if w.ndim > 1:
            w = w.mean(1)
        if sr != TARGET_SR:
            g = gcd(sr, TARGET_SR)
            w = resample_poly(w, TARGET_SR // g, sr // g)
        return w.astype(np.float32)

    def prep(x, pe=0.97):
        x = x - np.mean(x)
        x = np.append(x[0], x[1:] - pe * x[:-1])
        pk = np.max(np.abs(x))
        return x / pk if pk > 0 else x

    wp = [prep(load(p)) for p in df["path"]]
    ext = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
    mdl = (
        Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-base", output_hidden_states=True
        )
        .to(DEVICE)
        .eval()
    )
    embs = []
    with torch.no_grad():
        for w in wp:
            iv = ext(w, sampling_rate=TARGET_SR, return_tensors="pt").input_values.to(
                DEVICE
            )
            embs.append(mdl(iv).hidden_states[0].squeeze(0).mean(0).cpu().numpy())
    emb = np.stack(embs)
    spec = extract_spectral_features(wp, device=DEVICE)
    T = StandardScaler().fit_transform(emb)
    S = StandardScaler().fit_transform(spec)
    L = df["label"].to_numpy()
    np.savez(cf, emb_t=T, emb_s=S, labels=L)
    return T, S, L.astype(np.float32)


LOADERS = {"pcgita": prepare_pcgita, "neurovoz": prepare_neurovoz}
print("cargadores listos:", list(LOADERS))

cargadores listos: ['pcgita', 'neurovoz']


## 3 — Ejecución para ambos corpus

Para cada corpus: Optuna → estimación robusta (con y sin ICMM, umbral de Youden) →
figuras. Los resultados se guardan en `RESULTS` para el resumen final.

In [28]:
# ================================ EJECUCIÓN ================================
# Entrena una sola vez y CONGELA los resultados en results/. Si ya existen,
# los carga y dibuja las figuras desde ahí (deterministas, sin reentrenar).
import optuna, matplotlib.pyplot as plt, json as _js
from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
    train_test_split as _tts,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
RESULTS = {}
DATA = {}
SERIAL = {}
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
METRICS = ["auc", "acc", "sens", "spec"]


def _draw_roc_confusion(name, ot, op, th):
    roc_f, cm_f = FIGNAME[name]
    fpr, tpr, _ = roc_curve(ot, op)
    auc = roc_auc_score(ot, op)
    j = np.argmax(tpr - fpr)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, color="#2A9D8F", lw=2, label="GDBFNet + ICMM (AUC = %.3f)" % auc)
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.scatter(
        fpr[j], tpr[j], color="#E76F51", zorder=5, label="Youden (umbral = %.2f)" % th
    )
    ax.set_xlabel("1 - Especificidad")
    ax.set_ylabel("Sensibilidad")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, roc_f), dpi=200, facecolor="white", bbox_inches="tight"
    )
    plt.close()
    yp = (op >= th).astype(int)
    cm = confusion_matrix(ot, yp)
    fig, ax = plt.subplots(figsize=(4.2, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["HC", "EP"])
    ax.set_yticklabels(["HC", "EP"])
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")
    for a_ in range(2):
        for b_ in range(2):
            ax.text(
                b_,
                a_,
                int(cm[a_, b_]),
                ha="center",
                va="center",
                fontsize=14,
                color="white" if cm[a_, b_] > cm.max() / 2 else "black",
            )
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, cm_f), dpi=200, facecolor="white", bbox_inches="tight"
    )
    plt.close()
    return roc_f, cm_f


def _serial(name, R, n):
    def _ci(d, k):
        m, lo, hi = bootstrap_ci(d[k])
        return [round(float(m), 4), round(float(lo), 4), round(float(hi), 4)]

    return {
        "corpus": name,
        "arquitectura": "GDBFNet + ICMM",
        "n_sujetos": int(n),
        "hiperparametros": R["hp"],
        "mixup_alpha": float(R["mixup_alpha"]),
        "alpha_compuerta": float(R["alpha"]),
        "umbral_youden": float(R["oof_thr"]),
        "oof_auc": float(R["oof_auc"]),
        "metricas_sin_ICMM": {k: _ci(R["robust"]["base"], k) for k in METRICS},
        "metricas_con_ICMM": {k: _ci(R["robust"]["icmm"], k) for k in METRICS},
    }


def _train_corpus(name, T, S, L):
    def space(tr):
        return dict(
            lr=tr.suggest_float("lr", 1e-4, 5e-3, log=True),
            weight_decay=tr.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
            dropout=tr.suggest_float("dropout", 0.1, 0.5, step=0.05),
            label_smoothing=tr.suggest_float("label_smoothing", 0.0, 0.15, step=0.025),
            batch_size=tr.suggest_categorical("batch_size", [8, 16, 32]),
            dim_proj=tr.suggest_categorical("dim_proj", [64, 128, 256]),
            dim_hidden=tr.suggest_categorical("dim_hidden", [32, 64, 128]),
            mixup_alpha=tr.suggest_float("mixup_alpha", 0.1, 1.0, step=0.1),
        )

    def objective(tr):
        hp = space(tr)
        ma = hp.pop("mixup_alpha")
        skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        a = []
        for k, (t, v) in enumerate(skf.split(T, L)):
            yt, yp, _ = train_fold_regularized(
                make_train_ds(T, S, L, t, True, ma, RANDOM_STATE + k),
                DualBranchDataset(T, S, L, v),
                DEVICE,
                epochs=EPOCHS,
                patience=PATIENCE,
                **hp
            )
            a.append(roc_auc_score(yt, yp))
        return np.mean(a) - 0.5 * np.std(a)

    st = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    st.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    bp = st.best_params
    ma = bp.pop("mixup_alpha")
    bpm = {**bp, "epochs": EPOCHS, "patience": PATIENCE}
    rskf = RepeatedStratifiedKFold(
        n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=RANDOM_STATE
    )
    r = {"base": {m: [] for m in METRICS}, "icmm": {m: [] for m in METRICS}}
    al = []
    for k, (t, v) in enumerate(rskf.split(T, L)):
        vd = DualBranchDataset(T, S, L, v)
        yt, yp, _ = train_fold_regularized(
            make_train_ds(T, S, L, t, False, None, 0), vd, DEVICE, **bpm
        )
        m = youden_metrics(yt, yp)
        [r["base"][kk].append(m[kk]) for kk in METRICS]
        yt, yp, a = train_fold_regularized(
            make_train_ds(T, S, L, t, True, ma, RANDOM_STATE + k), vd, DEVICE, **bpm
        )
        m = youden_metrics(yt, yp)
        [r["icmm"][kk].append(m[kk]) for kk in METRICS]
        al.append(float(np.mean(a)))
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    ot, op, oal = [], [], []
    for k, (t, v) in enumerate(skf.split(T, L)):
        yt, yp, a = train_fold_regularized(
            make_train_ds(T, S, L, t, True, ma, RANDOM_STATE + k),
            DualBranchDataset(T, S, L, v),
            DEVICE,
            **bpm
        )
        ot.append(yt)
        op.append(yp)
        oal.append(np.ravel(a))
    ot = np.concatenate(ot)
    op = np.concatenate(op)
    oal = np.concatenate(oal)
    fpr, tpr, thr = roc_curve(ot, op)
    th = float(thr[np.argmax(tpr - fpr)])
    auc = float(roc_auc_score(ot, op))
    return bpm, ma, r, al, ot, op, oal, th, auc


def run_corpus(name):
    T, S, L = LOADERS[name]()
    DATA[name] = (T, S, L)
    path = os.path.join(RESULTS_DIR, "resultados_completos_%s.npz" % name)
    if os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        r = {
            "base": {k: list(d["base_%s" % k]) for k in METRICS},
            "icmm": {k: list(d["icmm_%s" % k]) for k in METRICS},
        }
        RESULTS[name] = {
            "hp": _js.loads(str(d["hp"])),
            "mixup_alpha": float(d["mixup_alpha"]),
            "robust": r,
            "alpha": float(d["alpha"]),
            "oof_auc": float(d["oof_auc"]),
            "oof_thr": float(d["oof_thr"]),
            "oof_true": d["oof_true"],
            "oof_prob": d["oof_prob"],
            "oof_alpha": d["oof_alpha"],
        }
        print("[%s] cargado de results/ (sin reentrenar)" % name)
    else:
        print("[%s] no hay resultados guardados -> entrenando..." % name)
        bpm, ma, r, al, ot, op, oal, th, auc = _train_corpus(name, T, S, L)
        RESULTS[name] = {
            "hp": bpm,
            "mixup_alpha": ma,
            "robust": r,
            "alpha": float(np.mean(al)),
            "oof_auc": auc,
            "oof_thr": th,
            "oof_true": ot,
            "oof_prob": op,
            "oof_alpha": oal,
        }
        np.savez(
            path,
            oof_true=ot,
            oof_prob=op,
            oof_alpha=oal,
            hp=_js.dumps(bpm),
            mixup_alpha=ma,
            alpha=np.mean(al),
            oof_auc=auc,
            oof_thr=th,
            **{"base_%s" % k: np.array(r["base"][k]) for k in METRICS},
            **{"icmm_%s" % k: np.array(r["icmm"][k]) for k in METRICS}
        )
        _tri, _vai = _tts(
            np.arange(len(L)), test_size=0.15, stratify=L, random_state=RANDOM_STATE
        )
        fm = fit_model(
            make_train_ds(T, S, L, _tri, True, ma, RANDOM_STATE),
            DualBranchDataset(T, S, L, _vai),
            DEVICE,
            **bpm
        )
        torch.save(
            {
                "corpus": name,
                "arquitectura": "DualBranchFusionNet (GDBFNet con compuerta) + ICMM",
                "state_dict": fm.state_dict(),
                "hiperparametros": bpm,
                "mixup_alpha": float(ma),
                "umbral_youden": th,
                "dim_t": int(T.shape[1]),
                "dim_s": int(S.shape[1]),
            },
            os.path.join(RESULTS_DIR, "modelo_gdbfnet_%s.pt" % name),
        )
        print(
            "[%s] guardado results/resultados_completos_%s.npz + modelo" % (name, name)
        )
    R = RESULTS[name]
    RESULTS[name]["figs"] = FIGNAME[name]
    SERIAL[name] = _serial(name, R, len(L))
    _js.dump(
        SERIAL[name],
        open(os.path.join(RESULTS_DIR, "resultados_gdbfnet_%s.json" % name), "w"),
        indent=2,
        ensure_ascii=False,
    )
    roc_f, cm_f = _draw_roc_confusion(name, R["oof_true"], R["oof_prob"], R["oof_thr"])
    print(
        "  figuras (desde predicciones guardadas) -> %s, %s | OOF AUC=%.3f alpha=%.3f"
        % (roc_f, cm_f, R["oof_auc"], R["alpha"])
    )


for c in CORPORA:
    run_corpus(c)
print("\nHecho.")

[pcgita] cargado de results/ (sin reentrenar)
  figuras (desde predicciones guardadas) -> res_roc.png, res_confusion.png | OOF AUC=0.921 alpha=0.757
[neurovoz] cargado de results/ (sin reentrenar)
  figuras (desde predicciones guardadas) -> res_roc_nv.png, res_confusion_nv.png | OOF AUC=0.914 alpha=0.464

Hecho.


## 4 — Resumen conjunto (para transcribir a la memoria)

In [29]:
# ================================ RESUMEN ================================
def fmt(d, k):
    m, lo, hi = bootstrap_ci(d[k])
    return f"{m:.3f} [{lo:.3f}, {hi:.3f}]"


NAME = {"pcgita": "PC-GITA", "neurovoz": "NeuroVoz"}
for c in CORPORA:
    R = RESULTS[c]
    r = R["robust"]
    print("=" * 72)
    print(f"  {NAME[c]} — GDBFNet (Youden coherente) | CV repetida + IC 95%")
    print("=" * 72)
    print(f"  {'Métrica':<14}{'GDBFNet':>26}{'GDBFNet + ICMM':>26}")
    print("  " + "-" * 66)
    for k, nm in zip(
        ["auc", "acc", "sens", "spec"],
        ["AUC", "Exactitud", "Sensibilidad", "Especificidad"],
    ):
        print(f"  {nm:<14}{fmt(r['base'],k):>26}{fmt(r['icmm'],k):>26}")
    s = np.mean(r["icmm"]["sens"])
    sp = np.mean(r["icmm"]["spec"])
    ac = np.mean(r["icmm"]["acc"])
    print("  " + "-" * 66)
    print(
        f"  alpha={R['alpha']:.3f} | coherencia: (sens+espec)/2={ (s+sp)/2:.3f} vs acc={ac:.3f} "
        f"| {'espec>sens' if sp>s else 'sens>espec'}"
    )
    print()

# Fila intra-corpus para tab:cc-main (AUC/Acc/Sens/Espec con GDBFNet + ICMM)
print("Filas intra-corpus (GDBFNet + ICMM) para tab:cc-main:")
for c in CORPORA:
    r = RESULTS[c]["robust"]["icmm"]
    print(
        f"  {NAME[c]:<9}",
        " & ".join(
            f"{np.mean(r[k]):.3f}".replace(".", ",")
            for k in ["auc", "acc", "sens", "spec"]
        ),
    )

  PC-GITA — GDBFNet (Youden coherente) | CV repetida + IC 95%
  Métrica                          GDBFNet            GDBFNet + ICMM
  ------------------------------------------------------------------
  AUC                 0.900 [0.880, 0.917]      0.898 [0.874, 0.920]
  Exactitud           0.879 [0.862, 0.894]      0.882 [0.861, 0.902]
  Sensibilidad        0.822 [0.782, 0.860]      0.822 [0.778, 0.862]
  Especificidad       0.936 [0.914, 0.956]      0.942 [0.920, 0.962]
  ------------------------------------------------------------------
  alpha=0.757 | coherencia: (sens+espec)/2=0.882 vs acc=0.882 | espec>sens

  NeuroVoz — GDBFNet (Youden coherente) | CV repetida + IC 95%
  Métrica                          GDBFNet            GDBFNet + ICMM
  ------------------------------------------------------------------
  AUC                 0.912 [0.894, 0.928]      0.910 [0.890, 0.927]
  Exactitud           0.886 [0.867, 0.903]      0.891 [0.873, 0.907]
  Sensibilidad        0.882 [0.858, 0.90

## 5 — Figuras derivadas de la corrida (fondo blanco)

Todas se generan a partir de los resultados calculados arriba, por lo que coinciden
con las tablas y el texto. Sufijo `_nv` para las de NeuroVoz.

In [30]:
# ---- alpha histograma + Mixup vs baseline ----
import matplotlib.pyplot as plt

TEAL = "#2A9D8F"
CORAL = "#E76F51"
BASE = "#4C72B0"
PURPLE = "#8E7CC3"
INK = "#16262B"
NAME = {"pcgita": "PC-GITA", "neurovoz": "NeuroVoz"}
SUF = {"pcgita": "", "neurovoz": "_nv"}

for c in CORPORA:
    R = RESULTS[c]
    a = R["oof_alpha"]
    suf = SUF[c]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.hist(a, bins=20, color=CORAL, alpha=0.85, edgecolor="white")
    ax.axvline(
        R["alpha"], color="black", ls="--", lw=1.5, label="media = %.3f" % R["alpha"]
    )
    ax.set_xlabel(r"$\alpha$ (peso de la rama temporal)")
    ax.set_ylabel("Frecuencia")
    ax.set_xlim(0, 1)
    ax.legend()
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_alpha_hist%s.png" % suf),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()

    r = R["robust"]
    metr = ["auc", "acc", "sens", "spec"]
    lab = ["AUC", "Accuracy", "Sensibilidad", "Especificidad"]

    def mlh(d, k):
        m, lo, hi = bootstrap_ci(d[k])
        return m, m - lo, hi - m

    bv = [mlh(r["base"], k) for k in metr]
    iv = [mlh(r["icmm"], k) for k in metr]
    x = np.arange(4)
    w = 0.38
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    ax.bar(
        x - w / 2,
        [v[0] for v in bv],
        w,
        yerr=[[v[1] for v in bv], [v[2] for v in bv]],
        capsize=4,
        color=BASE,
        alpha=0.85,
        label="Sin aumento (GDBFNet)",
        error_kw=dict(ecolor="#333", elinewidth=1.2, capthick=1.2),
    )
    ax.bar(
        x + w / 2,
        [v[0] for v in iv],
        w,
        yerr=[[v[1] for v in iv], [v[2] for v in iv]],
        capsize=4,
        color=CORAL,
        alpha=0.85,
        label="GDBFNet + ICMM",
        error_kw=dict(ecolor="#333", elinewidth=1.2, capthick=1.2),
    )
    ax.set_xticks(x)
    ax.set_xticklabels(lab)
    ax.set_ylim(0.7, 1.0)
    ax.legend()
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_mixup_vs_baseline%s.png" % suf),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()
    print("%s: res_alpha_hist%s.png, res_mixup_vs_baseline%s.png" % (NAME[c], suf, suf))

PC-GITA: res_alpha_hist.png, res_mixup_vs_baseline.png
NeuroVoz: res_alpha_hist_nv.png, res_mixup_vs_baseline_nv.png


In [31]:
# ---- Ablación de la fusión: ramas, concat simple y GDBFNet con compuerta ----
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler as SS
from sklearn.model_selection import cross_val_predict


def cv_auc(X, y):
    clf = make_pipeline(SS(), LogisticRegression(max_iter=2000, C=1.0))
    cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    p = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]
    return roc_auc_score(y, p)


for c in CORPORA:
    T, S, L = DATA[c]
    suf = SUF[c]
    at = cv_auc(T, L)
    asp = cv_auc(S, L)
    af = cv_auc(np.concatenate([T, S], axis=1), L)
    ag = float(RESULTS[c]["oof_auc"])  # GDBFNet + ICMM (compuerta aprendida)
    labels = [
        "Temporal\n(wav2vec2)",
        "Espectral\n(log-Mel)",
        "Concat.\nsimple",
        "GDBFNet\n(compuerta)",
    ]
    vals = [at, asp, af, ag]
    cols = [TEAL, PURPLE, "#9AA0A6", CORAL]
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    bars = ax.bar(labels, vals, color=cols, width=0.64)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            v + 0.006,
            "%.3f" % v,
            ha="center",
            fontsize=14,
            fontweight="bold",
            color=INK,
        )
    ax.axhline(0.5, ls="--", color="gray", lw=1)
    ax.set_ylabel("AUC")
    ax.set_ylim(0.5, 0.95)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_ablacion_fusion%s.png" % suf),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()
    print(
        "%s ablacion -> temporal=%.3f espectral=%.3f concat=%.3f GDBFNet=%.3f"
        % (NAME[c], at, asp, af, ag)
    )

PC-GITA ablacion -> temporal=0.867 espectral=0.662 concat=0.863 GDBFNet=0.921
NeuroVoz ablacion -> temporal=0.871 espectral=0.867 concat=0.914 GDBFNet=0.914


In [32]:
# ---- Efecto del numero de muestras ICMM por sujeto (n_aug) ----
# Se guarda/carga desde results/ para no reentrenar en cada ejecucion.
def mixup_n(T, S, L, tr, n, ma, seed):
    rng = np.random.default_rng(seed)
    ot, os_, ol = [T[tr]], [S[tr]], [L[tr]]
    for _ in range(n):
        at, as_, al = manifold_mixup_intraclass(T[tr], S[tr], L[tr], ma, rng)
        ot.append(at)
        os_.append(as_)
        ol.append(al)
    T2 = np.concatenate(ot)
    S2 = np.concatenate(os_)
    L2 = np.concatenate(ol)
    return DualBranchDataset(T2, S2, L2, np.arange(len(L2)))


for c in CORPORA:
    suf = SUF[c]
    npzf = os.path.join(RESULTS_DIR, "naug_%s.npz" % c)
    if os.path.exists(npzf):
        d = np.load(npzf)
        ns = d["ns"]
        means = d["means"]
        lo = d["lo"]
        hi = d["hi"]
        print("%s n_aug (cargado de results/)" % NAME[c])
    else:
        T, S, L = DATA[c]
        bpm = RESULTS[c]["hp"]
        ma = RESULTS[c]["mixup_alpha"]
        ns = [0, 1, 2, 3, 4]
        means = []
        lo = []
        hi = []
        for n in ns:
            skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
            a = []
            for k, (tr, v) in enumerate(skf.split(T, L)):
                yt, yp, _ = train_fold_regularized(
                    mixup_n(T, S, L, tr, n, ma, RANDOM_STATE + k),
                    DualBranchDataset(T, S, L, v),
                    DEVICE,
                    **bpm
                )
                a.append(roc_auc_score(yt, yp))
            m, l, h = bootstrap_ci(a)
            means.append(m)
            lo.append(l)
            hi.append(h)
        ns = np.array(ns)
        means = np.array(means)
        lo = np.array(lo)
        hi = np.array(hi)
        np.savez(npzf, ns=ns, means=means, lo=lo, hi=hi)
        print(
            "%s n_aug -> %s (guardado en results/)"
            % (NAME[c], [round(float(x), 3) for x in means])
        )
    errs = np.vstack([means - lo, hi - means])
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.errorbar(ns, means, yerr=errs, fmt="-o", color=CORAL, capsize=4, ecolor="#333")
    ax.set_xticks(ns)
    ax.set_xlabel("Muestras ICMM por sujeto")
    ax.set_ylabel("AUC")
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_naug%s.png" % suf),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()

PC-GITA n_aug (cargado de results/)
NeuroVoz n_aug (cargado de results/)


In [33]:
# ---- Lineas base: sensibilidad vs especificidad (desde baseline_results.json) ----
import json as _json

BL = os.path.join(PROJECT_ROOT, "results/baseline_results.json")
if os.path.exists(BL):
    d = _json.load(open(BL))
    order = sorted(d, key=lambda m: d[m]["auc_mean"])
    sens = [d[m]["sens_mean"] for m in order]
    spec = [d[m]["spec_mean"] for m in order]
    x = np.arange(len(order))
    w = 0.4
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.bar(x - w / 2, sens, w, label="Sensibilidad", color="#2A9D8F")
    ax.bar(x + w / 2, spec, w, label="Especificidad", color="#8E7CC3")
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=35, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend()
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_baselines_sens_spec.png"),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()
    print("res_baselines_sens_spec.png regenerada (fondo blanco)")
else:
    print("Copia baseline_results.json a PROJECT_ROOT para regenerar esa figura.")

res_baselines_sens_spec.png regenerada (fondo blanco)


In [34]:
# ---- Generalizacion entre corpus (fondo blanco) ----
labels = [
    "Intra\nPC-GITA",
    "Intra\nNeuroVoz",
    "PC-GITA->\nNeuro",
    "Neuro->\nPC-GITA",
    "Combinado->\nmixto",
]
intra_pc = (
    float(np.mean(RESULTS["pcgita"]["robust"]["icmm"]["auc"]))
    if "pcgita" in RESULTS
    else 0.898
)
intra_nv = (
    float(np.mean(RESULTS["neurovoz"]["robust"]["icmm"]["auc"]))
    if "neurovoz" in RESULTS
    else 0.910
)
TRANSFER = dict(pc2nv=0.602, nv2pc=0.652, mixed=0.887)  # del experimento cross-corpus
vals = [intra_pc, intra_nv, TRANSFER["pc2nv"], TRANSFER["nv2pc"], TRANSFER["mixed"]]
cols = [TEAL, TEAL, CORAL, CORAL, BASE]
fig, ax = plt.subplots(figsize=(7.6, 4.2))
b = ax.bar(labels, vals, color=cols, width=0.64)
for bar, v in zip(b, vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v + 0.012,
        "%.3f" % v,
        ha="center",
        fontsize=12,
        fontweight="bold",
        color=INK,
    )
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_ylabel("AUC")
ax.set_ylim(0.4, 1.0)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(
    os.path.join(IMAGES_DIR, "res_crosscorpus.png"),
    dpi=200,
    facecolor="white",
    bbox_inches="tight",
)
plt.close()
print("res_crosscorpus.png regenerada (intra %.3f / %.3f)" % (intra_pc, intra_nv))

res_crosscorpus.png regenerada (intra 0.898 / 0.910)


## 6 — Figuras pesadas (opcionales): codificadores y barrido por capas

Requieren extraer HuBERT/WavLM y las 13 capas de wav2vec 2.0. `MAKE_HEAVY=False` por defecto.

In [35]:
MAKE_HEAVY = False  # True -> regenera res_encoders y res_layer_auc (PC-GITA)
if MAKE_HEAVY:
    from src.preprocessing import load_waveforms, preprocess_waveform, load_metadata
    from transformers import AutoModel, AutoFeatureExtractor

    dfm = load_metadata(base_path="data")
    wr, dfm = load_waveforms(dfm, target_sr=TARGET_SR)
    wp = [preprocess_waveform(w) for w in wr]
    y = dfm["label"].to_numpy()

    def enc0(mid):
        ext = AutoFeatureExtractor.from_pretrained(mid)
        mdl = (
            AutoModel.from_pretrained(mid, output_hidden_states=True).to(DEVICE).eval()
        )
        out = []
        with torch.no_grad():
            for w in wp:
                iv = ext(
                    w, sampling_rate=TARGET_SR, return_tensors="pt"
                ).input_values.to(DEVICE)
                out.append(mdl(iv).hidden_states[0].squeeze(0).mean(0).cpu().numpy())
        return np.stack(out)

    encs = {
        "wav2vec2": "facebook/wav2vec2-base",
        "HuBERT": "facebook/hubert-base-ls960",
        "WavLM": "microsoft/wavlm-base",
    }
    au = {k: cv_auc(enc0(v), y) for k, v in encs.items()}
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.bar(list(au), list(au.values()), color=TEAL, width=0.55)
    ax.set_ylabel("AUC")
    ax.set_ylim(0.5, 0.95)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_encoders.png"),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()
    from src.embeddings import extract_multilayer_embeddings

    lay = list(range(13))
    em = extract_multilayer_embeddings(wp, "facebook/wav2vec2-base", lay, DEVICE)
    la = [cv_auc(em[i], y) for i in lay]
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(lay, la, "-o", color=TEAL, lw=2, mfc="white", mec=TEAL, mew=2)
    im = int(np.argmax(la))
    ax.scatter([im], [la[im]], s=150, color=CORAL, zorder=5)
    ax.set_xlabel("Capa (0 = codificador convolucional)")
    ax.set_ylabel("AUC")
    ax.set_xticks(lay)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(IMAGES_DIR, "res_layer_auc.png"),
        dpi=200,
        facecolor="white",
        bbox_inches="tight",
    )
    plt.close()
    print(
        "pesadas OK -> encoders=%s | capa max=%d (%.3f)"
        % ({k: round(v, 3) for k, v in au.items()}, im, la[im])
    )
else:
    print("MAKE_HEAVY=False -> res_encoders y res_layer_auc no regeneradas.")

MAKE_HEAVY=False -> res_encoders y res_layer_auc no regeneradas.


In [36]:
import json as _js

_js.dump(
    SERIAL,
    open(os.path.join(PROJECT_ROOT, "results/resultados_gdbfnet_todos.json"), "w"),
    indent=2,
    ensure_ascii=False,
)
print("Guardado resultados_gdbfnet_todos.json con:", list(SERIAL))

Guardado resultados_gdbfnet_todos.json con: ['pcgita', 'neurovoz']
